# מפגש 16 — הדגמה מודרכת: KNN שלנו, NumPy וספרייה
כבר למדנו לחזות ביד. כעת נפרק את הלמידה לפונקציות ונאמת תוצאה עם KNeighborsClassifier **רק אחרי** שהמימוש שלנו ברור. דוגמת מורה עם A–E.

## 1. תחזית פלט: מה ההבדל בין print ל־return?
קריאת print בתוך פונקציה רק מציגה; return מחזירה ערך לסביבה שקראה לה. חזו מהו אורך המרחק לפני הרצה.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier

def square_return(x):
    return x * x

def square_print(x):
    print("Value printed:", x * x)

returned = square_return(4)
only_printed = square_print(4)
print("Returned:", returned, "Print-function return:", only_printed)
assert returned == 16
assert only_printed is None

## 2. עוקבים אחר לולאה
בין [1,2] ל־[4,6] ריבועי ההפרשים הם 9 ו־16. רשמו את total אחרי כל איטרציה (9,25).

In [ ]:
def euclidean_distance(u, v):
    if len(u) != len(v):
        raise ValueError("Both vectors must have the same number of features")
    total = 0.0
    for j in range(len(u)):
        diff = float(u[j]) - float(v[j])
        total = total + diff ** 2
    return total ** 0.5
print('Loop distance:', euclidean_distance([1,2],[4,6]))
assert euclidean_distance([1,2],[4,6]) == 5.0
assert euclidean_distance([4,6],[1,2]) == 5.0

## 3. NumPy עושה בדיוק אותו חישוב
ההבדל הוא כתיבת פעולה על מערך שלם. np.sum מחבר את ריבועי ההפרשים; השורש נשאר אותו שורש.

In [ ]:
u = np.array([1.,2.])
v = np.array([4.,6.])
delta = u - v
squared = delta ** 2
distance_numpy = np.sqrt(np.sum(squared))
print("Delta:", delta, "Squares:", squared, "NumPy distance:",distance_numpy)
assert np.array_equal(delta,[-3.,-4.])
assert np.array_equal(squared,[9.,16.])
assert distance_numpy == 5.0

## 4. מסד דוגמה של מורה ושמירת מאפיין/תווית
X_train: חמש תצפיות עם שני מאפיינים; y_train: חמש תוויות Red/Blue. השאילתה q=[3,3] אינה מכילה תווית אמת.

In [ ]:
X_train = np.array([[3.,2.],[5.,3.],[5.,5.],[7.,3.],[3.,8.]])
y_train = np.array(["Red","Blue","Blue","Red","Red"])
query = np.array([3.,3.])
original_X = X_train.copy()
original_y = y_train.copy()
assert X_train.shape == (5,2)
assert y_train.shape == (5,)
assert query.shape == (2,)

## 5. מרחקים וקטוריים מכל השורות — משמעות axis=1
חיסור [3,3] מחמש שורות, ריבוע, סכימה **בכל שורה** ושורש. אנו מצפים לחמישה מספרים ולא לשניים.

In [ ]:
loop_distances = []
for row in X_train:
    loop_distances.append(euclidean_distance(row,query))
numpy_distances = np.sqrt(((X_train-query)**2).sum(axis=1))
print("Loop:", loop_distances)
print("NumPy:", numpy_distances)
assert np.allclose(loop_distances,[1.,2.,np.sqrt(8),4.,5.])
assert np.allclose(loop_distances,numpy_distances)
assert numpy_distances.shape == (5,)

## 6. פונקציית חיזוי עם מדיניות תיקו גלויה
שומרים distance ו־label באותה שורת DataFrame, ממיינים, בוחרים K ורק אז מצביעים. אם אין מנצחת יחידה — מחזירים None. אין לבחור את mode הראשונה בשתיקה.

In [ ]:
def predict_one(train_X, train_y, query, k):
    if not isinstance(k, int) or k < 1 or k > len(train_X):
        raise ValueError("K must be an integer from 1 to n_train")
    if len(train_X) != len(train_y):
        raise ValueError("Each training row must have one label")
    distances = []
    for row in train_X:
        distances.append(euclidean_distance(row, query))
    neighbors = pd.DataFrame({"distance": distances, "label": train_y})
    nearest = neighbors.sort_values("distance", kind="stable").head(k)
    modes = nearest["label"].mode().tolist()
    if len(modes) != 1:
        return None
    return modes[0]
for k in [1,3,5]:
    result = predict_one(X_train,y_train,query,k)
    print('Manual K=',k,'prediction=',result)
    assert result == {1:'Red',3:'Blue',5:'Red'}[k]

## 7. השוואת המימוש שלנו למסווג מוכן על אותה משימה
בתרגיל המקור זוהי **בדיקת התאמה**, לא החלפה של פונקציות התלמיד. משווים רק כאשר קבוצת השכנים והצבעת הקולות חד־משמעיות.

In [ ]:
for k in [1,3,5]:
    clf = KNeighborsClassifier(n_neighbors=k,weights="uniform",metric="euclidean")
    clf.fit(X_train,y_train)
    library_prediction = clf.predict([query])[0]
    manual_prediction = predict_one(X_train,y_train,query,k)
    print('K',k,'manual',manual_prediction,'library',library_prediction)
    assert manual_prediction == library_prediction
assert np.array_equal(X_train,original_X)
assert np.array_equal(y_train,original_y)

## 8. שתי שאילתות והבדלי מדיניות
ה־K של ההדגמה הוא 3. השאילתה [6,4] עשויה להכיל מרחקים שווים — אל תסיקו תוצאה עבור K=1 אוטומטית. K=3 בדוגמה מאפשר הצבעת רוב חד־משמעית.

In [ ]:
queries = np.array([[3.,3.],[6.,4.]])
manual_predictions = []
for q in queries:
    manual_predictions.append(predict_one(X_train,y_train,q,3))
library = KNeighborsClassifier(n_neighbors=3,weights="uniform",metric="euclidean")
library.fit(X_train,y_train)
library_predictions = library.predict(queries).tolist()
print('Two manual predictions',manual_predictions)
print('Two library predictions',library_predictions)
assert manual_predictions == library_predictions
assert manual_predictions == ['Blue','Blue']

## 9. תיקו קולות ומקרה K לא חוקי
כאן נבדוק מודעת למגבלה: K=3 עם תוויות A/B/C אינו מבטיח רוב. וכן אין לבקש K=7 כאשר חמש תצפיות בלבד.

In [ ]:
tie_X = np.array([[1.,0.],[0.,1.],[-1.,0.]])
tie_y = np.array(['A','B','C'])
tie_query = np.array([0.,0.])
tie_prediction = predict_one(tie_X,tie_y,tie_query,3)
print('Manual tie policy =>',tie_prediction)
assert tie_prediction is None
assert 7 > len(X_train)
assert euclidean_distance([0,0],[3,4]) == 5.

## 10. כרטיס יציאה
למה משתמשים ב־return ולא רק print? למה axis=1? מדוע לא ממיינים distances לחוד ו־labels לחוד? האם היעדר אמת לשאילתה מאפשר Accuracy? לא. מסווג Scikit-learn יילמד בהרחבה במפגש 17.